# Two Pointers — Complete Pattern Guide

Two Pointers is one of the **most frequently tested** patterns in coding interviews. It reduces O(n²) brute force to O(n) by using two indices that move through data in a coordinated way.

This lesson covers:
1. Why Two Pointers exists (the intuition)
2. The 6 Two Pointer sub-patterns
3. How to SPOT the pattern in an interview
4. Edge cases and gotchas
5. Interview execution framework

---
## 1. Why Two Pointers Exists

Many problems ask you to find pairs, triplets, or subarrays that satisfy some condition. The brute force approach uses nested loops:

```python
# Brute force: check all pairs → O(n²)
for i in range(len(arr)):
    for j in range(i+1, len(arr)):
        if condition(arr[i], arr[j]):
            # found something
```

Two Pointers eliminates the inner loop by using **structural guarantees** (usually sorting) to decide which pointer to move. Instead of checking all pairs, you check only the pairs that matter.

### The Core Insight

If the data is **sorted**, you can make intelligent decisions about which pointer to advance:
- Sum too small? Move the left pointer right (increase the sum)
- Sum too big? Move the right pointer left (decrease the sum)
- Found it? Record and move on

Each pointer moves **at most n times**, so the total work is O(n), not O(n²).

In [ ]:
# BRUTE FORCE: Find a pair with a given sum in a sorted array
def pair_sum_brute(nums, target):
    """O(n²) — check every pair"""
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return []

nums = [1, 2, 4, 6, 8, 11]
print("Brute force:", pair_sum_brute(nums, 10))  # [1, 4] → 2 + 8

In [ ]:
# TWO POINTER: Same problem, O(n)
def pair_sum_two_pointer(nums, target):
    """O(n) — two pointers from ends, array must be sorted"""
    left, right = 0, len(nums) - 1
    
    while left < right:
        current_sum = nums[left] + nums[right]
        
        if current_sum == target:
            return [left, right]
        elif current_sum < target:
            left += 1       # Need bigger sum → move left forward
        else:
            right -= 1      # Need smaller sum → move right backward
    
    return []

nums = [1, 2, 4, 6, 8, 11]
print("Two pointer:", pair_sum_two_pointer(nums, 10))  # [1, 4] → 2 + 8

### Why O(n)?

Each pointer moves in only one direction. `left` only moves right, `right` only moves left. Together they visit at most `n` positions total. No position is visited twice.

---
## 2. The 6 Two Pointer Sub-Patterns

| # | Pattern | Pointer Setup | Typical Use Case |
|---|---------|---------------|------------------|
| 1 | **Opposite Ends** | `left=0, right=n-1` | Pair sum, palindrome, container |
| 2 | **Same Direction (Slow/Fast)** | `slow=0, fast=0` | Remove duplicates, partition |
| 3 | **Sorted Array Multi-Sum** | Opposite ends inside a loop | 3Sum, 4Sum |
| 4 | **Partitioning / Dutch Flag** | Multiple pointers carving regions | Sort colors, move zeroes |
| 5 | **Merge Two Sorted** | One pointer per array | Merge sorted arrays/lists |
| 6 | **String Manipulation** | Left/right on string | Reverse, palindrome, squeeze |

---
## Pattern 1: Opposite Ends (Converging Pointers)

### When to use
- Array is **sorted** (or you can sort it)
- You need to find a **pair** satisfying a condition
- You're comparing elements from both ends

### Template
```python
left, right = 0, len(arr) - 1
while left < right:
    # Compute something with arr[left] and arr[right]
    if condition_met:
        # Record result, move both or one pointer
    elif need_bigger:
        left += 1
    else:
        right -= 1
```

### Key insight
Sorting gives you a **monotonic guarantee**: moving left increases, moving right decreases. This lets you eliminate entire regions of the search space with each step.

### Example: Two Sum II (sorted input)

In [ ]:
def two_sum_sorted(numbers, target):
    """LeetCode 167: Two Sum II - Input Array Is Sorted"""
    left, right = 0, len(numbers) - 1
    
    while left < right:
        s = numbers[left] + numbers[right]
        if s == target:
            return [left + 1, right + 1]  # 1-indexed
        elif s < target:
            left += 1
        else:
            right -= 1
    
    return []  # No solution found

# Test
print(two_sum_sorted([2, 7, 11, 15], 9))    # [1, 2]
print(two_sum_sorted([2, 3, 4], 6))          # [1, 3]
print(two_sum_sorted([-1, 0], -1))            # [1, 2]

### Example: Container With Most Water

This problem doesn't need a sorted array — the two pointer logic works because of a different guarantee: **we always move the shorter line** because keeping it can't improve the area.

In [ ]:
def max_area(height):
    """LeetCode 11: Container With Most Water"""
    left, right = 0, len(height) - 1
    best = 0
    
    while left < right:
        # Area = width × min height
        w = right - left
        h = min(height[left], height[right])
        best = max(best, w * h)
        
        # Move the shorter side — keeping it can't help
        if height[left] < height[right]:
            left += 1
        else:
            right -= 1
    
    return best

print(max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]))  # 49

**Why move the shorter side?**

The area is limited by `min(height[left], height[right])`. If we move the taller side inward, the width decreases and the height can only stay the same or decrease — so area can't improve. Moving the shorter side gives us a chance to find a taller line.

---
## Pattern 2: Same Direction (Slow/Fast Pointers)

### When to use
- Remove or skip elements **in place**
- Partition array into regions
- Collapse duplicates
- `slow` marks the write position, `fast` scans ahead

### Template
```python
slow = 0
for fast in range(len(arr)):
    if should_keep(arr[fast]):
        arr[slow] = arr[fast]
        slow += 1
# arr[:slow] contains the result
```

### Key insight
`slow` is always ≤ `fast`. Everything before `slow` is the "kept" region. `fast` explores the full array. You're building the answer in-place at the front.

### Example: Remove Duplicates from Sorted Array

In [ ]:
def remove_duplicates(nums):
    """LeetCode 26: Remove Duplicates from Sorted Array (in-place)"""
    if not nums:
        return 0
    
    slow = 0  # slow points to the last unique element
    
    for fast in range(1, len(nums)):
        if nums[fast] != nums[slow]:   # Found new unique element
            slow += 1
            nums[slow] = nums[fast]
    
    return slow + 1  # Length of unique portion

# Test
arr = [1, 1, 2, 2, 3, 4, 4, 5]
k = remove_duplicates(arr)
print(f"Unique count: {k}, Array: {arr[:k]}")  # 5, [1, 2, 3, 4, 5]

### Example: Move Zeroes

In [ ]:
def move_zeroes(nums):
    """LeetCode 283: Move Zeroes — move all 0s to end, maintain order of non-zeros"""
    slow = 0  # Next position to write a non-zero
    
    for fast in range(len(nums)):
        if nums[fast] != 0:
            nums[slow], nums[fast] = nums[fast], nums[slow]
            slow += 1
    
    return nums

print(move_zeroes([0, 1, 0, 3, 12]))  # [1, 3, 12, 0, 0]
print(move_zeroes([0, 0, 0]))           # [0, 0, 0]
print(move_zeroes([1]))                 # [1]

**Why swap instead of overwrite?**

Swapping preserves all elements. Overwriting would lose the value at `slow`. For Move Zeroes specifically, we could also do a two-pass (overwrite non-zeros, then fill zeros), but the swap approach is cleaner and more general.

### Example: Remove Element

In [ ]:
def remove_element(nums, val):
    """LeetCode 27: Remove Element — remove all instances of val in-place"""
    slow = 0
    
    for fast in range(len(nums)):
        if nums[fast] != val:
            nums[slow] = nums[fast]
            slow += 1
    
    return slow

arr = [3, 2, 2, 3]
k = remove_element(arr, 3)
print(f"Length: {k}, Array: {arr[:k]}")  # 2, [2, 2]

---
## Pattern 3: Sorted Array Multi-Sum (3Sum, 4Sum)

### When to use
- Find **triplets** or **quadruplets** that sum to a target
- Reduce from O(n³) or O(n⁴) to O(n²) or O(n³)

### Template (3Sum)
```python
nums.sort()  # MUST sort first
result = []
for i in range(len(nums) - 2):
    if i > 0 and nums[i] == nums[i-1]:  # Skip duplicates
        continue
    left, right = i + 1, len(nums) - 1
    while left < right:
        total = nums[i] + nums[left] + nums[right]
        if total == 0:
            result.append([nums[i], nums[left], nums[right]])
            # Skip duplicates for left and right
            while left < right and nums[left] == nums[left+1]: left += 1
            while left < right and nums[right] == nums[right-1]: right -= 1
            left += 1
            right -= 1
        elif total < 0:
            left += 1
        else:
            right -= 1
```

### Key insight
Fix one element with a for loop, then use Two Pointers on the remainder. This reduces one dimension: O(n³) → O(n²). For 4Sum, nest another for loop: O(n⁴) → O(n³).

### Example: 3Sum

In [ ]:
def three_sum(nums):
    """LeetCode 15: 3Sum — find all unique triplets that sum to 0"""
    nums.sort()
    result = []
    
    for i in range(len(nums) - 2):
        # Skip duplicate values for i
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        
        # Early termination: if smallest possible triple > 0, done
        if nums[i] > 0:
            break
        
        left, right = i + 1, len(nums) - 1
        
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            
            if total == 0:
                result.append([nums[i], nums[left], nums[right]])
                # Skip duplicates
                while left < right and nums[left] == nums[left + 1]:
                    left += 1
                while left < right and nums[right] == nums[right - 1]:
                    right -= 1
                left += 1
                right -= 1
            elif total < 0:
                left += 1
            else:
                right -= 1
    
    return result

print(three_sum([-1, 0, 1, 2, -1, -4]))  # [[-1, -1, 2], [-1, 0, 1]]
print(three_sum([0, 0, 0, 0]))             # [[0, 0, 0]]
print(three_sum([]))                        # []

### Step-by-step trace for 3Sum

Input: `[-1, 0, 1, 2, -1, -4]`

After sort: `[-4, -1, -1, 0, 1, 2]`

```
i=0, nums[i]=-4: left=1, right=5
  -4 + (-1) + 2 = -3 < 0 → left++
  -4 + (-1) + 2 = -3 < 0 → left++
  -4 + 0 + 2 = -2 < 0    → left++
  -4 + 1 + 2 = -1 < 0    → left++
  left >= right → done

i=1, nums[i]=-1: left=2, right=5
  -1 + (-1) + 2 = 0  ✓ FOUND! → append [-1, -1, 2]
  Skip dupes, left=3, right=4
  -1 + 0 + 1 = 0     ✓ FOUND! → append [-1, 0, 1]
  left=4, right=3 → done

i=2, nums[i]=-1: SKIP (duplicate of i=1)

i=3, nums[i]=0: > 0 check fails, 0 is not > 0
  left=4, right=5
  0 + 1 + 2 = 3 > 0 → right--
  left >= right → done

Result: [[-1, -1, 2], [-1, 0, 1]]
```

### The Duplicate Skipping Logic — Why It's Tricky

3Sum has **three places** where you skip duplicates:

1. **Outer loop (i):** `if i > 0 and nums[i] == nums[i-1]: continue`
   - Ensures we don't fix the same first element twice

2. **Left pointer:** `while left < right and nums[left] == nums[left+1]: left += 1`
   - After finding a match, skip identical left values

3. **Right pointer:** `while left < right and nums[right] == nums[right-1]: right -= 1`
   - After finding a match, skip identical right values

**Common mistake:** Only skipping at one level. You need ALL THREE to avoid duplicate triplets.

### Example: Three Sum Closest

In [ ]:
def three_sum_closest(nums, target):
    """LeetCode 16: 3Sum Closest — find triplet sum closest to target"""
    nums.sort()
    closest = float('inf')
    
    for i in range(len(nums) - 2):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        
        left, right = i + 1, len(nums) - 1
        
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            
            if abs(total - target) < abs(closest - target):
                closest = total
            
            if total < target:
                left += 1
            elif total > target:
                right -= 1
            else:
                return total  # Exact match
    
    return closest

print(three_sum_closest([-1, 2, 1, -4], 1))  # 2 → (-1 + 2 + 1)

---
## Pattern 4: Partitioning / Dutch National Flag

### When to use
- Categorize elements into 2-3 groups **in place**
- Sort array with only 2-3 distinct values
- Reorder based on a pivot

### Template (Dutch National Flag — 3 partitions)
```python
low, mid, high = 0, 0, len(arr) - 1
while mid <= high:
    if arr[mid] == 0:     # Category 1: goes to front
        arr[low], arr[mid] = arr[mid], arr[low]
        low += 1
        mid += 1
    elif arr[mid] == 1:   # Category 2: stays in middle
        mid += 1
    else:                 # Category 3: goes to back
        arr[mid], arr[high] = arr[high], arr[mid]
        high -= 1
        # DON'T increment mid — the swapped element needs checking
```

### Key insight
Three pointers maintain three regions: `[0..low-1]` is group 1, `[low..mid-1]` is group 2, `[high+1..n-1]` is group 3. The region `[mid..high]` is unexplored.

### Example: Sort Colors (Dutch National Flag)

In [ ]:
def sort_colors(nums):
    """LeetCode 75: Sort Colors — sort array of 0s, 1s, 2s in one pass"""
    low, mid, high = 0, 0, len(nums) - 1
    
    while mid <= high:
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1
            mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:  # nums[mid] == 2
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1
            # Note: mid does NOT increment here!
            # The element swapped from high hasn't been examined yet
    
    return nums

print(sort_colors([2, 0, 2, 1, 1, 0]))  # [0, 0, 1, 1, 2, 2]
print(sort_colors([2, 0, 1]))             # [0, 1, 2]
print(sort_colors([0]))                   # [0]

### Step-by-step trace for Sort Colors

Input: `[2, 0, 2, 1, 1, 0]`

```
low=0, mid=0, high=5

mid=0: nums[0]=2 → swap with high → [0, 0, 2, 1, 1, 2], high=4
mid=0: nums[0]=0 → swap with low  → [0, 0, 2, 1, 1, 2], low=1, mid=1
mid=1: nums[1]=0 → swap with low  → [0, 0, 2, 1, 1, 2], low=2, mid=2
mid=2: nums[2]=2 → swap with high → [0, 0, 1, 1, 2, 2], high=3
mid=2: nums[2]=1 → mid++          → mid=3
mid=3: nums[3]=1 → mid++          → mid=4
mid=4 > high=3 → DONE

Result: [0, 0, 1, 1, 2, 2] ✓
```

---
## Pattern 5: Merge Two Sorted Sequences

### When to use
- Merge two sorted arrays/lists into one
- Compare elements from two sorted sequences
- Find intersection/difference of sorted arrays

### Template
```python
i, j = 0, 0        # One pointer per array
result = []
while i < len(a) and j < len(b):
    if a[i] <= b[j]:
        result.append(a[i])
        i += 1
    else:
        result.append(b[j])
        j += 1
# Don't forget the remainder!
result.extend(a[i:])
result.extend(b[j:])
```

### Key insight
At every step, the smallest unprocessed element is at one of the two pointers. Just pick the smaller one and advance that pointer.

### Example: Merge Sorted Array (in-place)

In [ ]:
def merge_sorted(nums1, m, nums2, n):
    """LeetCode 88: Merge Sorted Array — merge nums2 into nums1 in-place
    
    Trick: fill from the END to avoid overwriting elements we still need.
    """
    p1 = m - 1       # Last real element in nums1
    p2 = n - 1       # Last element in nums2
    write = m + n - 1  # Write position (end of nums1)
    
    while p1 >= 0 and p2 >= 0:
        if nums1[p1] > nums2[p2]:
            nums1[write] = nums1[p1]
            p1 -= 1
        else:
            nums1[write] = nums2[p2]
            p2 -= 1
        write -= 1
    
    # If nums2 has remaining elements, copy them
    # (if nums1 has remaining, they're already in place)
    while p2 >= 0:
        nums1[write] = nums2[p2]
        p2 -= 1
        write -= 1
    
    return nums1

print(merge_sorted([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3))  # [1, 2, 2, 3, 5, 6]

**Why fill from the end?**

If we fill from the front, we'd overwrite elements in nums1 that we haven't processed yet. Filling from the back avoids this because the extra space (the zeros) is at the end.

### Example: Intersection of Two Sorted Arrays

In [ ]:
def sorted_intersection(a, b):
    """Find common elements in two sorted arrays (no duplicates in result)"""
    i, j = 0, 0
    result = []
    
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            # Avoid duplicates in result
            if not result or result[-1] != a[i]:
                result.append(a[i])
            i += 1
            j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    
    return result

print(sorted_intersection([1, 2, 2, 3, 4], [2, 2, 3, 5]))  # [2, 3]

---
## Pattern 6: String Manipulation with Two Pointers

### When to use
- **Reverse** a string/array in place
- Check if something is a **palindrome**
- Squeeze/compress strings
- Compare strings character by character from both ends

### Template (palindrome check)
```python
left, right = 0, len(s) - 1
while left < right:
    if s[left] != s[right]:
        return False
    left += 1
    right -= 1
return True
```

### Example: Valid Palindrome

In [ ]:
def is_palindrome(s):
    """LeetCode 125: Valid Palindrome — ignore non-alphanumeric, case-insensitive"""
    left, right = 0, len(s) - 1
    
    while left < right:
        # Skip non-alphanumeric characters
        while left < right and not s[left].isalnum():
            left += 1
        while left < right and not s[right].isalnum():
            right -= 1
        
        if s[left].lower() != s[right].lower():
            return False
        
        left += 1
        right -= 1
    
    return True

print(is_palindrome("A man, a plan, a canal: Panama"))  # True
print(is_palindrome("race a car"))                       # False
print(is_palindrome(" "))                                # True (empty after filtering)

### Example: Reverse String

In [ ]:
def reverse_string(s):
    """LeetCode 344: Reverse String — in place"""
    left, right = 0, len(s) - 1
    
    while left < right:
        s[left], s[right] = s[right], s[left]
        left += 1
        right -= 1
    
    return s

print(reverse_string(['h', 'e', 'l', 'l', 'o']))  # ['o', 'l', 'l', 'e', 'h']

### Example: Valid Palindrome II (remove at most one character)

In [ ]:
def valid_palindrome_ii(s):
    """LeetCode 680: Valid Palindrome II — can delete at most one character"""
    
    def is_palindrome_range(s, left, right):
        while left < right:
            if s[left] != s[right]:
                return False
            left += 1
            right -= 1
        return True
    
    left, right = 0, len(s) - 1
    
    while left < right:
        if s[left] != s[right]:
            # Try removing left char OR right char
            return (is_palindrome_range(s, left + 1, right) or
                    is_palindrome_range(s, left, right - 1))
        left += 1
        right -= 1
    
    return True

print(valid_palindrome_ii("aba"))     # True
print(valid_palindrome_ii("abca"))    # True (remove 'b' or 'c')
print(valid_palindrome_ii("abc"))     # False

---
## 3. How to SPOT Two Pointers in an Interview

This is the most critical skill. When you hear certain keywords or see certain problem structures, your brain should immediately think "Two Pointers."

### Decision Tree

```
Is the input sorted (or can you sort it)?
├── YES → Does it ask for pairs/triplets with a sum condition?
│         ├── Pairs → Pattern 1: Opposite Ends
│         └── Triplets/Quadruplets → Pattern 3: Multi-Sum
│
├── YES → Does it ask to merge two sorted sequences?
│         └── Pattern 5: Merge Two Sorted
│
├── Does it ask to modify array IN-PLACE?
│   ├── Remove duplicates → Pattern 2: Slow/Fast
│   ├── Move/remove specific values → Pattern 2: Slow/Fast
│   └── Sort 2-3 categories → Pattern 4: Dutch Flag
│
├── Is it about palindromes or string reversal?
│   └── Pattern 6: String Two Pointers
│
└── Does it involve two arrays/lists to compare?
    └── Pattern 5: Merge / compare
```

### Keyword → Pattern Mapping

| Keyword / Phrase | Pattern |
|------------------|---------|
| "sorted array" + "pair/sum" | Opposite Ends |
| "two sum" + "sorted" | Opposite Ends |
| "container", "water", "trapping" | Opposite Ends |
| "3Sum", "triplet", "three numbers" | Multi-Sum |
| "remove duplicates" + "in-place" | Slow/Fast |
| "move zeroes", "remove element" | Slow/Fast |
| "sort colors", "0s 1s 2s" | Dutch Flag |
| "partition", "rearrange" | Dutch Flag or Slow/Fast |
| "merge sorted" | Merge Two Sorted |
| "palindrome" | String Two Pointers |
| "reverse" + "in-place" | String Two Pointers |
| "subsequence" check | Same Direction |
| "O(1) extra space" + array modification | Slow/Fast or Dutch Flag |

### Two Pointers vs Other Patterns — How to Tell the Difference

| If you see... | It might be... | How to distinguish |
|---|---|---|
| Pair sum on **unsorted** array | HashMap (not Two Pointers) | Two Pointers needs sorted or structural order |
| **Subarray** sum / length | Sliding Window | Sliding window if you need a contiguous range |
| "Find pair" + sorted | Two Pointers | Classic opposite ends |
| Cycle in linked list | Fast/Slow Pointers | Different pattern (Floyd's cycle) |
| Subsequence check | Two Pointers (same direction) | One pointer per sequence |
| Binary search | Binary search, not Two Pointers | Though binary search uses two pointers, the logic is different (halving, not scanning) |

---
## 4. More Real-World Examples

### Is Subsequence

In [ ]:
def is_subsequence(s, t):
    """LeetCode 392: Is Subsequence
    
    Check if s is a subsequence of t.
    Two pointers moving in the same direction.
    """
    i, j = 0, 0  # i for s, j for t
    
    while i < len(s) and j < len(t):
        if s[i] == t[j]:
            i += 1  # Matched this char, move to next in s
        j += 1      # Always advance through t
    
    return i == len(s)  # Did we match all of s?

print(is_subsequence("abc", "ahbgdc"))  # True
print(is_subsequence("axc", "ahbgdc"))  # False
print(is_subsequence("", "anything"))    # True (empty is always a subsequence)

### Trapping Rain Water

In [ ]:
def trap(height):
    """LeetCode 42: Trapping Rain Water — two pointer approach, O(n) time O(1) space"""
    if not height:
        return 0
    
    left, right = 0, len(height) - 1
    left_max, right_max = height[left], height[right]
    water = 0
    
    while left < right:
        if left_max < right_max:
            left += 1
            left_max = max(left_max, height[left])
            water += left_max - height[left]
        else:
            right -= 1
            right_max = max(right_max, height[right])
            water += right_max - height[right]
    
    return water

print(trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))  # 6
print(trap([4, 2, 0, 3, 2, 5]))                       # 9

**Why Two Pointers works for Trapping Rain Water:**

Water at position `i` = `min(max_left, max_right) - height[i]`.

We don't need to precompute both max arrays. We always process the side with the **smaller** max first. Why? If `left_max < right_max`, then we know the water at `left` is bounded by `left_max` (because there's something at least as tall on the right). This is the same logic as Container With Most Water: **process the shorter side first**.

### Squares of a Sorted Array

In [ ]:
def sorted_squares(nums):
    """LeetCode 977: Squares of a Sorted Array
    
    Array is sorted but may have negatives.
    Largest square is at one of the two ends.
    Fill result from the back.
    """
    n = len(nums)
    result = [0] * n
    left, right = 0, n - 1
    write = n - 1  # Fill from the back (largest first)
    
    while left <= right:
        if abs(nums[left]) > abs(nums[right]):
            result[write] = nums[left] ** 2
            left += 1
        else:
            result[write] = nums[right] ** 2
            right -= 1
        write -= 1
    
    return result

print(sorted_squares([-4, -1, 0, 3, 10]))  # [0, 1, 9, 16, 100]
print(sorted_squares([-7, -3, 2, 3, 11]))  # [4, 9, 9, 49, 121]

---
## 5. Complexity Analysis

| Pattern | Time | Space | Notes |
|---------|------|-------|-------|
| Opposite Ends | O(n) | O(1) | Single pass, constant pointers |
| Slow/Fast | O(n) | O(1) | In-place modification |
| 3Sum | O(n²) | O(1)* | Sort is O(n log n), dominated by O(n²) outer loop × inner two-pointer. *O(n) for output |
| 4Sum | O(n³) | O(1)* | Same idea, one more nested loop |
| Dutch Flag | O(n) | O(1) | Single pass, 3 pointers |
| Merge Two Sorted | O(n + m) | O(n + m) | Or O(1) if merging in-place from back |
| String Palindrome | O(n) | O(1) | Two pointers moving inward |

**Key: Two Pointers almost always gives O(1) extra space.** That's a major advantage over HashMap-based solutions.

---
## 6. Common Gotchas

### Gotcha 1: Forgetting to sort
Two Pointer pair-sum only works on sorted arrays. If the input isn't sorted, you MUST sort first (or use a hash map).

In [ ]:
# WRONG: Two pointers on unsorted array
nums = [3, 1, 4, 1, 5, 9]
left, right = 0, len(nums) - 1
target = 10

# This will NOT work correctly!
while left < right:
    s = nums[left] + nums[right]
    if s == target:
        print(f"Found: {nums[left]} + {nums[right]}")
        break
    elif s < target:
        left += 1
    else:
        right -= 1
else:
    print("Not found (because array isn't sorted!)")

# CORRECT: Sort first
nums.sort()
print(f"\nSorted: {nums}")
# Now two pointers would work correctly

### Gotcha 2: while left < right vs while left <= right

- **`left < right`**: Use when pointers should NOT meet (pair problems, palindromes)
- **`left <= right`**: Use when pointers CAN be at the same position (Dutch Flag, Squares)

In [ ]:
# Demonstration: when it matters

# Palindrome: left < right (they shouldn't meet — middle char doesn't need checking)
s = "racecar"
left, right = 0, len(s) - 1
while left < right:
    print(f"Comparing s[{left}]='{s[left]}' with s[{right}]='{s[right]}'")
    left += 1
    right -= 1
print("Middle char 'e' not compared — correct for palindrome!\n")

# Squares: left <= right (need to process the element when pointers meet)
nums = [-2, 0, 3]
left, right = 0, 2
result = []
while left <= right:
    if abs(nums[left]) > abs(nums[right]):
        result.append(nums[left] ** 2)
        left += 1
    else:
        result.append(nums[right] ** 2)
        right -= 1
print(f"Squares (reversed): {result}")  # [9, 4, 0] — need the zero!

### Gotcha 3: Not incrementing mid after swapping with high (Dutch Flag)

When you swap `arr[mid]` with `arr[high]`, the new element at `mid` is unknown (it came from the unexplored region). You must examine it again. Only increment `mid` when you swap with `low` or when the element is already in the correct category.

### Gotcha 4: Duplicate handling in 3Sum

You must skip duplicates at THREE levels:
1. The outer loop (`i`)
2. The left pointer (after finding a match)
3. The right pointer (after finding a match)

Missing any one level → duplicate triplets in the result.

### Gotcha 5: Sorting destroys original indices

If the problem asks for **original indices** (like Two Sum on LeetCode), you can't sort and use two pointers — you'd lose the original positions. Use a HashMap instead.

**Rule of thumb:**
- Problem returns **values** → sort + two pointers ✓
- Problem returns **indices** → usually HashMap

---
## 7. Interview Execution Framework

When you identify Two Pointers in an interview, follow this sequence:

### Step 1: Clarify
- "Is the array sorted?" (If not, can I sort it?)
- "Do you need indices or values?" (Determines if sorting is allowed)
- "Can there be duplicates?" (Affects duplicate-skipping logic)
- "Should I modify in-place or return a new array?"

### Step 2: State the brute force
"The brute force would check all pairs/triplets — O(n²)/O(n³). Since the array is sorted, I can use two pointers to reduce this to O(n)/O(n²)."

### Step 3: Explain the pointer strategy
"I'll place one pointer at the start and one at the end. If the sum is too small, I'll move the left pointer right to increase it. If too large, I'll move the right pointer left."

### Step 4: Handle edge cases before coding
- Empty array
- Single element
- All duplicates
- All negative / all positive
- No valid answer exists

### Step 5: Code it

### Step 6: Trace through an example
Walk through your code with a small input.

### Step 7: State complexity
"Time: O(n). Space: O(1). Each pointer moves at most n times."

---
## 8. Two Pointers vs Sliding Window

These two patterns are closely related and often confused. Here's how to tell them apart:

| Feature | Two Pointers | Sliding Window |
|---------|-------------|----------------|
| **Pointer direction** | Often converging (opposite ends) | Always same direction (left follows right) |
| **What you're finding** | Pairs, triplets, partitions | Subarrays, substrings |
| **When to shrink** | Based on comparison (too big/small) | When window becomes invalid |
| **Sorting** | Usually required | Usually not needed |
| **In-place modification** | Common | Rare |
| **Contiguous subarray** | No (comparing distant elements) | Yes (always a contiguous window) |

**Simple rule:** If you need a **contiguous subarray/substring**, think Sliding Window. If you need **pairs from different parts** of the array or **in-place modification**, think Two Pointers.

---
## 9. Summary

| Pattern | Setup | Move Logic | Classic Problems |
|---------|-------|------------|------------------|
| Opposite Ends | `L=0, R=n-1` | Sum too small → L++, too big → R-- | Two Sum II, Container |
| Slow/Fast | `slow=0, for fast` | Keep? Write at slow++ | Remove Duplicates, Move Zeroes |
| Multi-Sum | Fix i, two-pointer rest | Same as opposite ends inside loop | 3Sum, 3Sum Closest |
| Dutch Flag | `low=0, mid=0, high=n-1` | 0→swap low, 1→mid++, 2→swap high | Sort Colors |
| Merge | `i=0, j=0` | Pick smaller, advance that pointer | Merge Sorted Array |
| String | `L=0, R=n-1` | Compare ends, move inward | Palindrome, Reverse |

### The Three Things That Make Two Pointers Work:
1. **Sorted order** (or a structural invariant like Container With Most Water)
2. **Monotonic pointer movement** (each pointer only moves in one direction)
3. **Pruning guarantee** (moving a pointer eliminates candidates that can't be optimal)